<a href="https://colab.research.google.com/github/suder54ul/LLMP/blob/main/module10_handson1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 - Hands-on 1: RAG in Production (Local Stack)
**Large Language Models in Production**
**Open – May 2026**

**Facilitator:** A/P TAN Wee Kek  
**Email:** distwk@nus.edu.sg

**Learning Objectives** (from Module 9)
- Explain how RAG improves factual grounding of LLM applications
- Implement the full 6-step RAG pipeline using Ollama + LLaMA 3.1 8B + LangChain + ChromaDB
- Compare LLM responses **with RAG** vs **without RAG**
- Understand local RAG architecture and production considerations

## 🔧 Ollama Setup Instructions (Must be done before proceeding)

Start a new terminal and run the following command to pull the required embeddings model with Ollama:

`ollama pull nomic-embed-text`<br>
`ollama list`

## 🔧 LangSmith Setup Instructions (Recommended)

LangSmith will trace the full RAG pipeline (retrieval + generation). Use the same API key as before.

After running, visit [smith.langchain.com](https://smith.langchain.com/) → Tracing: `NUS_LLMP_RAG_Exercise` to inspect traces.

In [ ]:
# === FORCE DISABLE LANGSMITH TRACING (Run this cell first) ===
# import os

# # Explicitly disable tracing
# os.environ["LANGCHAIN_TRACING_V2"] = "false"

# # Remove any leftover LangSmith keys if they exist
# for key in list(os.environ.keys()):
#     if key.startswith("LANGCHAIN_"):
#         del os.environ[key]

# print("LangSmith tracing has been explicitly disabled.")

In [ ]:
# === SETUP (run first) ===
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# === LANG SMITH MONITORING ENABLED ===
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "ur_langsmith_api_key_here"   # ← PASTE YOUR KEY HERE
os.environ["LANGSMITH_PROJECT"] = "NUS_LLMP_Evaluation_Exercise"

llm = ChatOllama(model="llama3.1:8b", temperature=0.0, num_ctx=8192)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Setup complete – Ollama + LLaMA 3.1 8B + ChromaDB + Local Embeddings ready!")

## Part 1: Substantive RAG Documents (MAS / Singapore Financial Services)

In [ ]:
# Four realistic policy documents for RAG demonstration
docs = [
    Document(page_content="""Excerpt from MAS Guidelines on Responsible Use of Generative AI in Financial Services (Feb 2026)
Financial institutions must adopt a risk-based approach. Key risks include data privacy/PDPA compliance, bias and fairness, explainability, cybersecurity, hallucinations leading to regulatory non-compliance, and the need for human oversight in high-risk decisions.""", metadata={"source": "mas_ai_guidelines.pdf", "department": "Compliance", "date": "2026-02"}),

    Document(page_content="""PDPA Compliance Policy – Customer Personal Data
All customer personal data must be protected under the Personal Data Protection Act. Organisations must obtain consent, limit collection to necessary purposes, ensure data accuracy, and implement reasonable security arrangements. Retention of personal data is limited to as long as it is necessary for the stated purpose.""", metadata={"source": "pdpa_policy.pdf", "department": "Legal", "date": "2025-11"}),

    Document(page_content="""Customer Transaction Records Retention Policy
Customer transaction records must be retained for a minimum of 7 years after account closure or the last transaction, whichever is later. Longer retention may be required for regulatory investigations or legal holds. Records must be stored securely and deleted once the retention period expires unless otherwise required.""", metadata={"source": "retention_policy.pdf", "department": "Operations", "date": "2026-01"}),

    Document(page_content="""Cybersecurity Requirements for AI Systems
AI systems must implement robust input validation, prompt injection detection, and continuous monitoring against adversarial attacks. All AI-generated outputs must be logged with audit trails. Institutions must conduct regular penetration testing and maintain incident response plans specific to generative AI risks.""", metadata={"source": "cybersecurity_ai.pdf", "department": "IT Security", "date": "2026-03"})
]

print(f"Loaded {len(docs)} substantive documents for RAG demonstration.")

## Part 2: The 6-Step RAG Ingestion Pipeline

In [ ]:
# Delete vector store if it already exists to ensure a clean slate for this demo
if os.path.exists("chroma_db"):

    import shutil
    shutil.rmtree("chroma_db")
    print("Existing ChromaDB vector store deleted for a clean demo.")

In [ ]:
# Step 1–4: Loading → Splitting → Embedding → Vector Store
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="mas_financial_rag",
    persist_directory="./chroma_db"
)

# The following retriever will be used in the next step for RAG question-answering. It retrieves the top 3 most relevant chunks based on cosine similarity.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Indexed {len(splits)} chunks into ChromaDB vector store.")

In [ ]:
# Demonstrate how to inspect the content in the vector store (for debugging/verification purposes)
# This is not part of the main RAG flow but can be useful to see what was indexed.
# Retrieve all documents in the collection to verify indexing
all_docs = vectorstore.get(include=["documents", "embeddings", "metadatas"])

for i in range(len(all_docs['documents'])):

    print(f"Document {i+1}:")
    print(f"Content: {all_docs['documents'][i]}")
    print(f"Embedding: {all_docs['embeddings'][i]}")
    print(f"Metadata: {all_docs['metadatas'][i]}")
    print("-" * 50)

## Part 3: Baseline – LLM Without RAG

In [ ]:
query = "According to our policies, how long must we retain customer transaction records?"

no_rag_prompt = ChatPromptTemplate.from_template("Answer the question:\n\nQuestion: {q}")
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

response_no_rag = no_rag_chain.invoke({"q": query})
print("=== WITHOUT RAG ===\n")
print(response_no_rag)

## Part 4: Basic RAG Chain (Step 5–6: Retrieval + Generation)

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the provided context. "
    "If you cannot answer from the context, say 'I don't know based on the provided documents.'\n\n"
    "Context:\n{context}\n\nQuestion: {q}"
)

def format_docs(docs_list):
    return "\n\n---\n\n".join(doc.page_content for doc in docs_list)

rag_chain = (
    {"context": retriever | format_docs, "q": lambda x: x}
    | rag_prompt
    | llm
    | StrOutputParser()
)

response_rag = rag_chain.invoke(query)
print("=== WITH RAG ===\n")
print(response_rag)

## Part 5: Inspect Retrieved Context (Debugging RAG)

In [ ]:
# Inspect the top retrieved documents for the query to understand what context the model had access to when generating the RAG response.
# How many documents were retrieved, and what information did they contain? This can help explain the model's answer and identify any gaps in the retrieved context.
# Which one of the fours documents was not retrived and why?

retrieved_docs = retriever.invoke(query)
print("=== RETRIEVED CONTEXT ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1} (source: {doc.metadata['source']}):\n{doc.page_content}\n---\n")

## Part 6: Your Turn + Group Discussion

**Exercise:**
1. Try different queries (e.g., about data privacy, cybersecurity, or bias in AI).
2. Compare the answers with and without RAG again.
3. Modify chunk size or `k` value and observe changes in retrieval quality.

**Group Discussion Questions:**
- How did RAG change the quality and trustworthiness of the answer?
- What risks did you observe even with RAG?
- How would you productionise this (scaling, governance, evaluation)?

**Suggested Further Practice Use Cases (try in your groups):**
- Internal HR policy Q&A system
- Product knowledge base for customer support
- Regulatory compliance checker for financial advisors
- Contract review assistant

**Key Takeaway:** RAG dramatically improves factual grounding, but retrieval quality, governance, and evaluation remain critical for production use.